# Beatrix diffusion experiments on Colab

This notebook installs [alephllm-diffusion-experiments](https://github.com/AbstractEyes/alephllm-diffusion-experiments) at
`v0.3.1` and runs the experiments in sessions. Each session is one call; it skips every step already done, so running it again
after a stop or a lost runtime picks up where it left off.

| session | what runs | about |
|---|---|---|
| 1 | the mount checks, the mount read, the port check, the restart test, stage 1 of the grid, the picture test's stage A (64 pictures) | 2.5-3 h |
| 2 | the grid of record (612 cells), the pick, the mount grids, the mount contrast, the export, publishing the grids | 4-7 h |
| 3 | the picture test's stage B (about 900 pictures) | 1-1.5 h |
| 4 | the hub sliders: Beatrix's slider features at three readings (her stream at a phrase's closing stop, her hub's blackboard, both), then seven slider arms trained by Anima's own objective and judged on held-out scenes (about 1,600 pictures; the render batch is timed first) | 1.5-2.5 h |
| 5 | the slider at the dual-extraction read's pick: Beatrix's slider features at block 20 (her stream at a phrase's closing stop), then three slider arms: the neutral phrases kept off both sides of the slider in training, its untrained control, and the same reading with session 4's recipe (about 700 pictures) | 1-1.5 h |

Sessions 4 and 5 need nothing from sessions 1-3 or from each other: on a new machine run cell 1, then cell 2 with `SESSION = 4`
or `SESSION = 5`.

**Before running.** Runtime > Change runtime type > RTX PRO 6000. In the Secrets panel (the key icon) add `HF_TOKEN`, a token
with read and write access to `AbstractPhil/beatrix-diffusion-colab` (private: the reference data and every session's files) and
write access to `AbstractPhil/geolip-beatrix-anima-data` and `AbstractPhil/geolip-beatrix-anima` (where the grids and the
pictures are published), and let this notebook read it. The token is never printed; the compute steps run without it.

**Reading the output.** Each step prints a line when it starts, a progress line every five minutes, and a line when it ends.
Every step's full output is kept in `runs/logs/` and goes to the private repo with the session's files.

**Stopping.** Press the stop button of the running cell once: a running grid stops at its next cell and keeps the cells it has
(any other step finishes first). Run the session cell again to continue. The long steps also save their files every half hour,
so a lost runtime costs at most half an hour: on a new machine, run cell 1, then cell 2 with the same session number.


In [ ]:
# CELL 1 - INSTALL (once per machine; about 10 minutes): the package at v0.3.1, the diffusion-pipe fork and its ComfyUI,
# and the pinned environment in this runtime's Python; then the environment check (the card, every pin, the libraries' commits).
import os, subprocess, sys
TAG = "v0.3.1"
ROOT = "/content/alephllm-diffusion-experiments"
def sh(cmd, cwd=ROOT):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, cwd=cwd).returncode
if not os.path.exists(ROOT):
    assert sh(f"git clone --quiet --branch {TAG} --depth 1 https://github.com/AbstractEyes/alephllm-diffusion-experiments {ROOT}",
              cwd="/content") == 0
else:                                   # a checkout from an earlier release: move it to this release's tag
    assert sh(f"git fetch --quiet --depth 1 origin tag {TAG}") == 0
    assert sh(f"git checkout --quiet {TAG}") == 0
assert sh("git submodule update --init --quiet external/diffusion-pipe") == 0
assert sh("git -C external/diffusion-pipe submodule update --init --quiet submodules/ComfyUI") == 0
assert sh(f"{sys.executable} -m pip install --quiet -r requirements.txt") == 0
ok = sh(f"{sys.executable} -m alephllm_diffusion.environment") == 0
print("READY: run cell 2" if ok else "THE ENVIRONMENT DIFFERS FROM THE PINS (above): run this cell again; if it persists, the run stops at its start")


In [ ]:
# CELL 2 - RUN A SESSION: 1, then 2, then 3; 4 and 5 (the sliders) each on its own. Running it again resumes the session.
SESSION = 5
import importlib, sys
sys.path.insert(0, "/content/alephllm-diffusion-experiments/src"); importlib.invalidate_caches()
from alephllm_diffusion import sessions
sessions.run(SESSION)


In [ ]:
# CELL 3 - STATUS: every step of every session, done (when, on which card) or not yet, and the state of the two grids.
import importlib, sys
sys.path.insert(0, "/content/alephllm-diffusion-experiments/src"); importlib.invalidate_caches()
from alephllm_diffusion import sessions
sessions.status(fetch=True)
